# atari_50m_endpoint

In [ ]:
# Import both experiments' configs, the results loader and the analysis functions.
import importlib.util
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from analysis.curves import return_curve, subsample
from analysis.stats import mean_ci
from main import load_results

from config import EXPERIMENT


def load_experiment(path):
    spec = importlib.util.spec_from_file_location(path.parent.name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.EXPERIMENT


DETERMINISTIC = load_experiment(Path("../atari_50m_deterministic/config.py"))
PLOTS_DIR = EXPERIMENT.results_dir.parent / "plots"

In [ ]:
# Load DQN's stored runs per game from this experiment and atari_50m_deterministic.
SOURCES = {
    "DQN, atari_50m_deterministic": (DETERMINISTIC, "dqn_atari"),
    "DQN": (EXPERIMENT, "dqn_atari"),
}
component = EXPERIMENT.component("dqn_atari")
GAMES = component.sweep["ENV_HYPERS.GAME"]
FRAMESKIP = component.config.ENV_HYPERS.FRAMESKIP

results = {}
for agent, (experiment, name) in SOURCES.items():
    for game in GAMES:
        runs = load_results(experiment, name, where={"ENV_HYPERS.GAME": game})
        if runs is not None:
            results[agent, game] = runs

In [ ]:
# Compute each agent's mean return per game with a 95% bootstrap CI.
curves = {}
for (agent, game), runs in results.items():
    timesteps, returns = subsample(return_curve(runs.reward, runs.done))
    curves[agent, game] = (timesteps * FRAMESKIP, mean_ci(returns), len(returns))

In [ ]:
# Style for the per-game learning-curve plots.
GAME_STYLE = {
    "figure.figsize": (9, 6),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 20,
    "axes.labelsize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 16,
    "legend.frameon": False,
    "legend.loc": "upper left",
}
COLORS = {"DQN": "tab:green", "DQN, atari_50m_deterministic": "tab:grey"}
# The earlier run is drawn first and wider, so an exact match stays visible.
LINE_WIDTHS = {"DQN": 1.75, "DQN, atari_50m_deterministic": 4.0}
BAND_ALPHA = 0.2
LEGEND_LABEL = "{agent} (n={runs})"
XLABEL = "Frames"
YLABEL = "Return"
YLABEL_KW = {"va": "center", "labelpad": 20}
XTICKS = np.array([10, 20, 30, 40, 50]) * 1_000_000
XTICK_LABELS = [f"{tick // 1_000_000}M" for tick in XTICKS]


def title(game):
    return game.replace("_", " ").title()

In [ ]:
# Plot one learning curve per game: this experiment's DQN against the earlier run.
def draw(ax, agent, game):
    frames, (mean, low, high), runs = curves[agent, game]
    color = COLORS[agent]
    ax.fill_between(frames, low, high, color=color, alpha=BAND_ALPHA)
    label = LEGEND_LABEL.format(agent=agent, runs=runs)
    ax.plot(frames, mean, color=color, linewidth=LINE_WIDTHS[agent], label=label)


figures = {}
for game in GAMES:
    agents = [agent for agent in SOURCES if (agent, game) in curves]
    if not agents:
        continue
    with plt.rc_context(GAME_STYLE):
        fig, ax = plt.subplots()
        for agent in agents:
            draw(ax, agent, game)
        ax.set(title=title(game), xlabel=XLABEL)
        ax.set_ylabel(YLABEL, **YLABEL_KW)
        ax.set_xticks(XTICKS, XTICK_LABELS)
        ax.legend()
        fig.tight_layout()
    figures[f"dqn_{game}"] = fig
plt.show()

In [ ]:
# Style for the all-games grid, laid out as in atari_50m_deterministic.
TOTAL_FRAMES = component.config.AGENT_HYPERS.TOTAL_TIMESTEPS * FRAMESKIP
GRID_STYLE = {
    "figure.figsize": (16, 9.5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 20,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 20,
    "legend.frameon": False,
    "figure.labelsize": 20,
}
GRID_SHAPE = (3, 4)
PANELS = {  # (column, row) of each game's panel
    "battle_zone": (0, 0),
    "double_dunk": (0, 1),
    "name_this_game": (0, 2),
    "phoenix": (1, 0),
    "qbert": (1, 1),
    "pong": (1, 2),
    "breakout": (2, 0),
    "seaquest": (2, 1),
    "centipede": (2, 2),
    "ms_pacman": (3, 0),
    "beam_rider": (3, 1),
    "space_invaders": (3, 2),
}
PANEL_ASPECT = 3 / 4
GRID_XTICKS = [0, TOTAL_FRAMES]
GRID_XTICK_LABELS = ["0", f"{TOTAL_FRAMES / 1_000_000:.0f}M"]
GRID_LAYOUT = {"rect": (0.03, 0.06, 1, 1), "w_pad": 0.2, "h_pad": 0.2}
GRID_XLABEL = {"t": "Frames", "y": 0.055, "va": "top"}
GRID_YLABEL = {"t": "Return"}
GRID_LEGEND = {"loc": "upper center", "bbox_to_anchor": (0.5, 0.035), "ncol": 2}

In [ ]:
# Plot every game in one grid: this experiment's DQN against the earlier run.
# Seed counts differ per game, so the shared legend names only the agents.
grids = {}
if curves:
    with plt.rc_context(GRID_STYLE):
        fig, axes = plt.subplots(*GRID_SHAPE)
        legend = {}
        for game, (column, row) in PANELS.items():
            ax = axes[row, column]
            ax.set_box_aspect(PANEL_ASPECT)
            drawn = [agent for agent in SOURCES if (agent, game) in curves]
            if not drawn:
                ax.axis("off")
                continue
            for agent in drawn:
                draw(ax, agent, game)
            ax.set_title(title(game))
            ax.set_xticks(GRID_XTICKS, GRID_XTICK_LABELS)
            for agent, handle in zip(drawn, ax.get_lines(), strict=True):
                legend[agent] = handle
        fig.tight_layout(**GRID_LAYOUT)
        fig.supxlabel(**GRID_XLABEL)
        fig.supylabel(**GRID_YLABEL)
        agents = [agent for agent in SOURCES if agent in legend]
        fig.legend([legend[agent] for agent in agents], agents, **GRID_LEGEND)
    grids["grid_dqn"] = fig
figures.update(grids)
plt.show()

In [ ]:
# Save every plotted figure as a PDF, and the grid also as a PNG, in the plots dir.
PLOTS_DIR.mkdir(exist_ok=True)
for name, fig in figures.items():
    fig.savefig(PLOTS_DIR / f"{name}.pdf", bbox_inches="tight")
for name, fig in grids.items():
    fig.savefig(PLOTS_DIR / f"{name}.png", bbox_inches="tight")
print(f"saved {len(figures)} plot(s) to {PLOTS_DIR}")